# Naive Text Generator Without Transformer


## The Predictive Text Frequency Map


Intuition: Create sub-phrases of text and track the frequency distribution of the next word that after each the sub-phrase.


Construct a lookup map where each unique sub-phrase serves as a key, mapping to a list of the words that immediately follow it in the sentence.

Example 1:\
Input: "I am coming"\
Contiguous Sub-Phrases: "I", "I am", "I am coming.", "am", "am coming.", "coming."\
Lookup Map: \

```json
{
  "I": { "am": 1 },
  "I am": { "coming": 1 },
  "I am coming.": {},
  "am": { "coming": 1 },
  "am coming.": {},
  "coming.": {}
}
```


Example 2: \
Input: "I think therefore I am. I think I know."\
Lookup Map:\

```Json
{
  "I": { "think": 2, "am.": 1, "know.": 1 },
  "I think": { "therefore": 1, "I": 1 },
  "I think therefore": { "I": 1 },
  "I think therefore I": { "am.": 1 },
  "I think therefore I am.": { "I": 1 },
  "I think therefore I am. I": { "think": 1 },
  "I think therefore I am. I think": { "I": 1 },
  "I think therefore I am. I think I": { "know.": 1 },
  "I think therefore I am. I think I know.": {},
  "think": { "therefore": 1, "I": 1 },
  "think therefore": { "I": 1 },
  "think therefore I": { "am.": 1 },
  "think therefore I am.": { "I": 1 },
  "think therefore I am. I": { "think": 1 },
  "think therefore I am. I think": { "I": 1 },
  "think therefore I am. I think I": { "know.": 1 },
  "think therefore I am. I think I know.": {},
  "therefore": { "I": 1 },
  "therefore I": { "am.": 1 },
  "therefore I am.": { "I": 1 },
  "therefore I am. I": { "think": 1 },
  "therefore I am. I think": { "I": 1 },
  "therefore I am. I think I": { "know.": 1 },
  "therefore I am. I think I know.": {},
  "am.": { "I": 1 },
  "am. I": { "think": 1 },
  "am. I think": { "I": 1 },
  "am. I think I": { "know.": 1 },
  "am. I think I know.": {},
  "know.": {}
}
```


In [1]:
from collections import defaultdict


def build_sub_phrase_cache(predictive_cache: dict[str, defaultdict[str, int]], text: str) -> None:
    # Split the text by spaces
    split_texts = text.split(" ")
    n = len(split_texts)

    # Add a sentinel at the end to control the iteration
    split_texts.append("#END#")

    # Build the cache
    for i in range(0, n):
        sub_phrase = split_texts[i]
        for j in range(i + 1, n + 1):
            # sub_phrase is the key into the first dictionary
            # split_texts[j] is the key into the inner dictionary
            # We increment the value by one.
            # With default dict, the value of a new entry into the inner dictionary will be 0 + 1
            predictive_cache[sub_phrase][split_texts[j]] += 1
            sub_phrase += " " + split_texts[j]

In [2]:
import json


def print_predictive_cache(predictive_cache: dict[str, defaultdict[str, int]]):
    print(json.dumps(predictive_cache))

In [3]:
predictive_cache = defaultdict(lambda: defaultdict(int))
build_sub_phrase_cache(predictive_cache, "I am coming")
print_predictive_cache(predictive_cache)

{"I": {"am": 1}, "I am": {"coming": 1}, "I am coming": {"#END#": 1}, "am": {"coming": 1}, "am coming": {"#END#": 1}, "coming": {"#END#": 1}}


In [4]:
build_sub_phrase_cache(predictive_cache, "I think therefore I am. I think I know.")
print_predictive_cache(predictive_cache)

{"I": {"am": 1, "think": 2, "am.": 1, "know.": 1}, "I am": {"coming": 1}, "I am coming": {"#END#": 1}, "am": {"coming": 1}, "am coming": {"#END#": 1}, "coming": {"#END#": 1}, "I think": {"therefore": 1, "I": 1}, "I think therefore": {"I": 1}, "I think therefore I": {"am.": 1}, "I think therefore I am.": {"I": 1}, "I think therefore I am. I": {"think": 1}, "I think therefore I am. I think": {"I": 1}, "I think therefore I am. I think I": {"know.": 1}, "I think therefore I am. I think I know.": {"#END#": 1}, "think": {"therefore": 1, "I": 1}, "think therefore": {"I": 1}, "think therefore I": {"am.": 1}, "think therefore I am.": {"I": 1}, "think therefore I am. I": {"think": 1}, "think therefore I am. I think": {"I": 1}, "think therefore I am. I think I": {"know.": 1}, "think therefore I am. I think I know.": {"#END#": 1}, "therefore": {"I": 1}, "therefore I": {"am.": 1}, "therefore I am.": {"I": 1}, "therefore I am. I": {"think": 1}, "therefore I am. I think": {"I": 1}, "therefore I am. I

## Text Generation


In [5]:
import secrets


def get_next_word(freq_map: dict[str, int]) -> str:
    total_weight = sum(freq_map.values())
    # randbelow will return between 0 to N-1
    target = secrets.randbelow(total_weight)
    cumulative = 0
    result = None

    for word, weight in freq_map.items():
        cumulative += weight
        if target < cumulative:
            result = word
            break

    return result

In [6]:
def generate_text(cache: dict[str, dict[str, int]], phrase: str, max_num_words: int = 10):
    words = [phrase]
    for _ in range(max_num_words):
        # Lookup the phrase and breakout of the loop if phrase does not exist
        freq_map = cache.get(phrase, None)
        if not freq_map:
            break

        next_word = get_next_word(freq_map)
        if not next_word:
            break

        words.append(next_word)
        phrase += " " + next_word

    return " ".join(words)

## Let's test it out


In [7]:
phrase = "I"
print(generate_text(predictive_cache, phrase, 30))

I am coming #END#


In [8]:
class DoubleRollingHasher:
    """
    Manages a stateful polynomial rolling hash across a sequence of words.
    Uses two distinct prime bases and modulos to prevent collisions, completely
    eliminating the need to store raw sub-phrase string keys in memory.
    """

    def __init__(self):
        # Two entirely independent pairs of large primes
        self.B1, self.M1 = 313, 10**9 + 7
        self.B2, self.M2 = 619, 10**9 + 9
        self.hash1 = 0
        self.hash2 = 0

    def append(self, word: str) -> tuple[int, int]:
        """Appends a word token to the rolling sequence safely."""
        # Python's built-in hash can be negative; make it positive
        raw_id = abs(hash(word))

        # Apply the modulo defensively to the token ID for each independent bucket
        word_id_m1 = raw_id % self.M1
        word_id_m2 = raw_id % self.M2

        # Calculate both polynomial rolling transformations safely
        self.hash1 = (self.hash1 * self.B1 + word_id_m1) % self.M1
        self.hash2 = (self.hash2 * self.B2 + word_id_m2) % self.M2

        return (self.hash1, self.hash2)

    def reset(self):
        """Resets the hash states back to zero for the start of a new window chain."""
        self.hash1 = 0
        self.hash2 = 0

In [9]:
from typing import Tuple


def build_sub_phrase_cache_hash(
    predictive_cache: dict[Tuple[int, int], defaultdict[str, int]],
    dbl_hash: DoubleRollingHasher,
    text: str,
) -> None:
    # Split the text by spaces
    split_texts = text.split(" ")
    n = len(split_texts)

    # Add a sentinel at the end to control the iteration
    split_texts.append("#END#")

    # Build the cache
    for i in range(0, n):
        dbl_hash.reset()
        current_hash = dbl_hash.append(split_texts[i])
        for j in range(i + 1, n + 1):
            predictive_cache[current_hash][split_texts[j]] += 1
            current_hash = dbl_hash.append(" " + split_texts[j])

In [10]:
def generate_text_hash(
    cache: dict[Tuple[int, int], defaultdict[str, int]],
    dbl_hash: DoubleRollingHasher,
    phrase: str,
    max_num_words: int = 10,
):
    dbl_hash.reset()
    words = [phrase]
    current_hash = dbl_hash.append(phrase)

    for _ in range(max_num_words):
        # Lookup the phrase and breakout of the loop if phrase does not exist
        freq_map = cache.get(current_hash, None)
        if not freq_map:
            break

        next_word = get_next_word(freq_map)
        if not next_word:
            break

        words.append(next_word)
        current_hash = dbl_hash.append(" " + next_word)

    return " ".join(words)

In [11]:
def print_cache_json(cache: dict[Tuple[int, int], defaultdict[str, int]]):
    # Convert tuple keys to strings and defaultdicts to normal dicts for JSON
    readable_cache = {f"Hash {k}": dict(v) for k, v in cache.items()}

    print(json.dumps(readable_cache, indent=4))

In [12]:
predictive_cache = defaultdict(lambda: defaultdict(int))
dbl_hash = DoubleRollingHasher()
build_sub_phrase_cache_hash(predictive_cache, dbl_hash, "I am coming")
# print_cache_json(predictive_cache)

build_sub_phrase_cache_hash(predictive_cache, dbl_hash, "I think therefore I am. I think I know.")
print_cache_json(predictive_cache)

{
    "Hash (275309300, 313192565)": {
        "am": 1,
        "think": 2,
        "am.": 1,
        "know.": 1
    },
    "Hash (255677624, 564875433)": {
        "coming": 1
    },
    "Hash (831513903, 135938996)": {
        "#END#": 1
    },
    "Hash (785923322, 514309763)": {
        "coming": 1
    },
    "Hash (798416215, 835789554)": {
        "#END#": 1
    },
    "Hash (610704409, 889677016)": {
        "#END#": 1
    },
    "Hash (472421325, 800860269)": {
        "therefore": 1,
        "I": 1
    },
    "Hash (384472782, 247790609)": {
        "I": 1
    },
    "Hash (726129081, 207068319)": {
        "am.": 1
    },
    "Hash (990011970, 665325368)": {
        "I": 1
    },
    "Hash (259893595, 661081818)": {
        "think": 1
    },
    "Hash (647305695, 144305932)": {
        "I": 1
    },
    "Hash (992830276, 150053832)": {
        "know.": 1
    },
    "Hash (695057414, 37275002)": {
        "#END#": 1
    },
    "Hash (717003047, 188878470)": {
        "therefor

In [13]:
phrase = "I"
print(generate_text_hash(predictive_cache, dbl_hash, phrase, 30))

I think therefore I am. I think I know. #END#


Reference: https://s201.q4cdn.com/141608511/files/doc_financials/2026/q4/10K-NVDA.pdf

In [14]:
text = """
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is now a data center scale AI infrastructure
company reshaping all industries.
Our technology stack includes the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs, as well as hundreds of domain-specific
software libraries, frameworks, algorithms, software development kits, or SDKs, and application programming interfaces, or APIs. This deep and broad software
stack accelerates the performance and facilitates the deployment of NVIDIA accelerated computing for computationally intensive workloads such as artificial
intelligence, or AI, model training and inference, data analytics, scientific computing, robotics, and 3D graphics, with vertical-specific optimizations to address
industries ranging from healthcare and telecom to automotive and manufacturing.
Introduced with the Blackwell architecture, our data-center-scale offerings feature extreme co-design where the infrastructure’s chips, networking, systems,
software, and algorithms are holistically architected and optimized to maximize performance and scale. Hundreds of thousands of GPUs can be interconnected
to function as a single giant computer. This type of data center architecture and scale is needed for the development and deployment of modern AI and
accelerated computing applications.
The GPU was initially used to simulate human imagination, enabling the virtual worlds of video games and films. Today, it also simulates human intelligence,
enabling a deeper understanding of language, science, and the physical world. Its parallel processing capabilities, supported by tens of thousands of computing
cores, are essential for deep learning algorithms. This form of AI, in which software writes itself by learning from large amounts of data, can serve as the brain of
computers, robots, and self-driving cars that can perceive, understand and reason about the world. GPU-powered AI solutions are being developed by
thousands of enterprises to deliver services and products that would have been immensely difficult or even impossible with traditional coding. Examples include
generative AI, which can create new content such as text, code, images, audio, video, molecule structures, and recommendation systems; and agentic AI where
systems of AI models work in concert to automatically complete a task.
NVIDIA has a platform strategy, bringing together hardware, systems, software, algorithms, libraries, AI models and training data sets, and services to create
unique value for the markets we serve. While the computing requirements of these end markets are diverse, we address them with a unified underlying
programmable architecture allowing us to support several multi-billion-dollar end markets with the same underlying technology by using a variety of software
stacks developed either internally or by third-party developers and partners. The large and growing number of developers and installed base across our
platforms strengthens our ecosystem and increases the value of our platform for our customers.
Innovation is at our core. We have invested over $76.7 billion in research and development since our inception, yielding inventions that are essential to modern
computing. Our invention of the GPU in 1999 sparked the growth of the PC gaming market and redefined computer graphics. With our introduction of CUDA in
2006, we opened the parallel processing capabilities of our GPU to a broad range of compute-intensive applications, paving the way for the emergence of
modern AI. In 2012, the AlexNet neural network, trained on NVIDIA GPUs, won the ImageNet computer image recognition competition, marking the “Big Bang”
moment of AI. We introduced our first Tensor Core GPU in 2017, built from the ground-up for the new era of AI, and our first autonomous driving system-onchips, or SoC, in 2018. Our acquisition of Mellanox in 2020 expanded our offerings to include networking, enabled our platforms to be data center scale, and led
to the introduction of a new processor class – the data processing unit, or DPU. Over the past 5 years, we have built full software stacks that run on top of our
GPUs and CUDA to bring AI to the world’s largest industries, including NVIDIA DRIVE stack for autonomous driving, Clara for healthcare, Omniverse for physical
AI applications, and NVIDIA AI Enterprise software – essentially an operating system for enterprise AI applications. In 2023, we introduced our first data center
CPU, Grace, built for giant-scale AI and high-performance computing, or HPC. In 2024, we launched the NVIDIA Blackwell architecture – connecting 36 Grace
CPUs and 72 Blackwell GPUs in a data center scale, liquid-cooled design – for real-time trillion-parameter inference and training. In fiscal year 2026, we
launched and scaled the NVIDIA Blackwell Ultra platform, optimized for agentic, reasoning, and physical AI. Building on the architectural breakthroughs of
Blackwell and leveraging Dynamo inference software, it delivers a significant increase in token throughput and reduction in cost per token compared to the
Hopper generation. More recently, in support of market development, we have accelerated the release cadence of our open AI model platforms including NVIDIA
Nemotron for agentic AI and Cosmos for physical AI. With a strong engineering culture, we drive fast, yet harmonized, product and technology innovations in all
dimensions of computing including silicon, systems, networking, software and algorithms. More than half of our engineers work on software.
All major cloud service providers, or CSPs, AI model makers, and enterprises use our data center-scale infrastructure and computing platforms to accelerate the
services and offerings they deliver to billions of end users and customers, including AI solutions and assistants, AI foundation models, advertising, search,
recommendation systems, social networking, data processing, online shopping, live video, and translation. AI model makers use our infrastructure and software hosted at CSPs to develop, build
and run AI models, product offerings, and services.
Enterprises and startups across a broad range of industries use our accelerated computing platforms to build new generative and agentic AI-enabled products
and services, and/or to dramatically accelerate and reduce the costs of their workloads and workflows. The enterprise software industry uses them for new AI
assistants, chatbots, and agents; the transportation industry for autonomous driving; the healthcare industry for accelerated and computer-aided drug discovery;
and the financial services industry for customer support and fraud detection.
Researchers and developers use our computing solutions to accelerate a wide range of important applications, from simulating molecular dynamics to climate
forecasting. With support for 6,000 applications, NVIDIA computing enables some of the most promising areas of discovery, from climate prediction to materials
science and from wind tunnel simulation to genomics. Including GPUs and networking, NVIDIA powers over 78% of the supercomputers on the global TOP500
list, including 9 of the top 10 systems on the Green500 list.
Gamers choose NVIDIA GPUs to enjoy immersive, increasingly cinematic virtual worlds. In addition to serving the growing number of gamers, the market for PC
GPUs is expanding because of the growing population of live streamers, broadcasters, artists, and creators. With the advent of generative and agentic AI, we
expect a broader set of PC users to choose NVIDIA GPUs for running these applications locally on their PC, which is critical for privacy, latency, and costsensitive AI applications.
Professional artists, architects and designers use NVIDIA partner products accelerated with our GPUs and software platform for a range of creative, engineering,
and design use cases, such as creating visual effects in movies or designing buildings and products. In addition, generative and agentic AI is expanding the
market for our workstation-class GPUs, as more enterprise customers develop and deploy AI applications with their data on-premises.
Headquartered in Santa Clara, California, NVIDIA was incorporated in California in April 1993 and reincorporated in Delaware in April 1998.
"""

In [21]:
predictive_cache = defaultdict(lambda: defaultdict(int))
build_sub_phrase_cache_hash(predictive_cache, dbl_hash, text)
print(generate_text_hash(predictive_cache, dbl_hash, "software", 100))

software and algorithms. More than half of our engineers work on software.
All major cloud service providers, or CSPs, AI model makers, and enterprises use our data center-scale infrastructure and computing platforms to accelerate the
services and offerings they deliver to billions of end users and customers, including AI solutions and assistants, AI foundation models, advertising, search,
recommendation systems, social networking, data processing, online shopping, live video, and translation. AI model makers use our infrastructure and software hosted at CSPs to develop, build
and run AI models, product offerings, and services.
Enterprises and startups across a broad range of industries use our accelerated computing platforms to


## Text Chunking

Our first approach to storing sub-phrases is computationally expensive and does not scale well for large texts. For example, processing a text containing 100,000 words would be slow and could easily exhaust available memory.

* **Space complexity:** O(KLN²)
* **Time complexity:** O(LN²)

Where:

* **K** is the average number of words cached for each sub-phrase.
* **L** is the average length of a sub-phrase.
* **N** is the total number of words in the input text.

By introducing a rolling hash, we significantly reduced both the processing time and the storage required for representing each sub-phrase. Although the runtime improved dramatically, the overall storage requirement remained quadratic because every possible sub-phrase was still represented.

* **Space complexity:** O(KN²)
* **Time complexity:** O(N)

To further improve memory efficiency, we introduce **chunking**. Rather than processing the entire text as a single, massive sequence, we divide it into smaller chunks of a fixed maximum size. Each chunk is processed independently while retaining the resulting sub-phrases and their corresponding next-word distributions.

A chunk containing at most **C** words can generate at most **O(C²)** unique sub-phrases. Storing the next-word distributions for these sub-phrases therefore requires **O(KC²)** space. Given **M** chunks, the total complexity becomes:

* **Space complexity:** O(KMC²)
* **Time complexity:** O(N)

Where:

* **K** is the average number of words cached for each sub-phrase.
* **C** is the maximum number of words in a chunk.
* **M** is the number of chunks.
* **N** is the total number of words in the input text.

Since the chunk size **C** is fixed, the number of chunks is proportional to the size of the input:

$$
MC \approx N
$$

Substituting this relationship gives:

$$
O(KMC^2) = O(KCN)
$$

Because **C** is a constant, the asymptotic complexities simplify to:

* **Space complexity:** **O(KN)**
* **Time complexity:** **O(N)**

The progression of the three approaches is summarized below:

| Approach                | Space     | Time     |
| ----------------------- | --------- | -------- |
| Naive                   | O(KLN²)   | O(LN²)   |
| Rolling hash            | O(KN²)    | O(N)     |
| Rolling hash + chunking | **O(KN)** | **O(N)** |

Rolling hash eliminates the quadratic processing time by enabling constant-time hash updates as the window slides across the text. Chunking complements this optimization by reducing the number of sub-phrases that must be stored, lowering the storage complexity from quadratic to linear. Together, these optimizations produce an algorithm whose time and space requirements both grow linearly with the size of the input text.


### Chunking Strategy

Naive chunking splits text at fixed boundaries (e.g., every *C* words or characters). While this reduces memory usage, it breaks structural integrity:

* paragraphs get split mid-thought
* tables get fragmented row-by-row
* code blocks and lists lose meaning
* semantic context is destroyed across boundaries

To address this, we adopt **structure-aware chunking**, which respects document layout before applying size constraints.

**Core Idea**

Instead of treating the input as a flat sequence of words, we first decompose it into **semantic blocks**:

* Paragraphs
* Tables
* Lists
* Code blocks (if present)
* Sentences (as a fallback unit)

We then build chunks by **packing whole blocks together** up to a maximum size **C**.

**Step 1: Parse Document into Blocks**

We first segment the document into structural units:

* Paragraph boundaries: newline-separated text blocks
* Tables: detected via markup (e.g., Markdown pipes `|`, HTML `<table>`)
* Lists: grouped list items are kept together
* Code blocks: fenced blocks (\` \` ) are treated as atomic

Each block becomes the smallest indivisible unit for chunking.

**Step 2: Estimate Block Size**

For each block, we estimate its size in tokens/words:

* Paragraph → number of words
* Table → number of cells or serialized words
* List → total words across items
* Code → tokenized length or word count

This gives us:

$$
\text{size(block)} \rightarrow w_i
$$

**Step 3: Greedy Packing into Chunks**

We then construct chunks using a greedy strategy:

* Initialize an empty chunk
* Add blocks sequentially
* If adding a block exceeds limit **C**, start a new chunk

This ensures:

* No structural unit is split
* Each chunk size is bounded by **C**
* Context within a block is preserved

**Step 4: Handling Oversized Blocks**

Some blocks (e.g., very large tables or long code files) may exceed **C** on their own.

In that case:

* Tables → split row-wise (not cell-wise)
* Code → split by function/class boundary if possible
* Paragraphs → fallback to sentence-level splitting

**Algorithm Summary**

1. Parse document → blocks
2. Compute block sizes
3. Greedily pack blocks into chunks up to size **C**
4. Split only when a single block exceeds **C**
5. Process each chunk independently